<span style="color:red; font-family:Helvetica Neue, Helvetica, Arial, sans-serif; font-size:2em;">An Exception was encountered at '<a href="#papermill-error-cell">In [6]</a>'.</span>

In [1]:
# Định nghĩa các giá trị mặc định (Papermill sẽ đè lên các giá trị này)
input_surface = "default_path.stl"
E = 0.01
nu = 0.09
cell_size = 8

In [2]:
# Parameters
input_surface = "D:/3_Aboard/Project/modeling/stl/13x8.STL"
E = 0.01
nu = 0.09
cell_size = 8.0


In [3]:
import json, sys, copy, os, igl
import numpy as np
import meshio

def is_inside(v, f, p):
    winds = igl.winding_number(v, f, p)
    mask = np.array(np.rint(winds), dtype=int)
    
    return np.array([e % 2 == 1 for e in mask])

def change_file_extension(filename, new_extension):
  base, ext = os.path.splitext(filename)
  return base + "." + new_extension if ext else filename + "." + new_extension

In [4]:
notebook_dir = os.path.dirname(os.path.abspath(__file__)) if '__file__' in globals() else os.getcwd()
microstructure_repo_path = notebook_dir
matopt_repo_path = os.path.join(os.path.dirname(microstructure_repo_path), "matopt")

# input_surface = input_surface = "/mnt/d/3_Aboard/Project/modeling/stl/8cube.STL" #"/mnt/c/Users/Asus/desktop/Part1.STL" # USER: Change this to your STL file path"C:\Users\ASUS\Desktop\Part1.STL"
stitch_cells_cli = os.path.join(microstructure_repo_path, "build/isosurface_inflator/cut_cells_cli")
only_cube_cells = False

In [5]:
# E = 0.01
# nu = 0.09
# cell_size = 8
# k = 3
# out_path = str(E) + "_" + str(cell_size) + ".obj"
out_path = f"{input_surface[:-4]}_{E}_{cell_size}.obj_{only_cube_cells}.obj"
print(out_path)

m = meshio.read(input_surface)
input_surface = change_file_extension(input_surface, 'obj')

v = m.points.astype(float)

# --- ĐOẠN XỬ LÝ CĂN GIỮA VÀ ÉP LƯỚI ĐỐI XỨNG ---
# 1. Dời trọng tâm của file 3D về chính xác gốc tọa độ (0,0,0)
center = (np.amax(v, axis=0) + np.amin(v, axis=0)) / 2.0
v = v - center
m.points = v
m.write(input_surface) # Ghi đè lại file để phần mềm C++ lõi nhận đúng tọa độ tâm

# 2. Tính toán Bounding Box mới đã được căn giữa
bbox = [np.amin(v, axis=0), np.amax(v, axis=0)]

# 3. Ép buộc số lượng ô lưới phải đối xứng hai bên trục (từ -N đến +N)
max_dims = np.amax(np.abs(bbox), axis=0)
corner1 = list(map(int, np.ceil(max_dims / cell_size)))
corner0 = [-c for c in corner1] 
# -----------------------------------------------------

print("Góc lưới đối xứng:", corner0, corner1)
#f = m.cells.data


# use return E for uniform Young's modulus
# use i,j,k to vary the Young's modulus through the dimensions
def young(i, j, k):
    # return E
    print(k)
    if k == 0:
        return 0.001
    if k == 1:
        return 0.0015
    if k == 2:
        return 0.002
    if k == 3:  
        return 0.005
    if k == 4:
        return 0.005
    if k == 5:
        return 0.005
    if k == 6:
        return 0.005
    if k == 7:
        return 0.005
    if k == 8:
        return 0.005
    if k == 9:
        return 0.005
    return E
    

D:/3_Aboard/Project/modeling/stl/13x8_0.01_8.0.obj_False.obj
Góc lưới đối xứng: [-1, -1, -1] [1, 1, 1]


<span id="papermill-error-cell" style="color:red; font-family:Helvetica Neue, Helvetica, Arial, sans-serif; font-size:2em;">Execution using papermill encountered an exception here and stopped:</span>

In [6]:
sys.path.insert(0, os.path.join(matopt_repo_path, 'tools/material2geometry'))
from material2geometry import Material2Geometry

ModuleNotFoundError: No module named 'material2geometry'

In [ ]:
mat2geo = Material2Geometry(in_path=os.path.join(matopt_repo_path, "tools/material2geometry/0646_geo_1_coeffs.txt"))

In [ ]:
patterns = []
entry = {"params": [],
"symmetry": "Cubic",
"pattern": os.path.join(microstructure_repo_path, "data/patterns/3D/reference_wires/pattern0646.wire"),
"index": [0,0,0]}

for i in range(corner0[0], 1 + corner1[0]):
    for j in range(corner0[1], 1 + corner1[1]):
        for k in range(corner0[2], 1 + corner1[2]):
            geo_params = mat2geo.evaluate(nu, young(i, j, k))
            entry["params"] = geo_params
            entry["index"] = [i,j,k]
            patterns.append(copy.deepcopy(entry))

with open("data.json", 'w') as f:
    json.dump(patterns, f)

In [ ]:
os.system(stitch_cells_cli + " -p data.json " +
           (("--surface " + input_surface) if not only_cube_cells else "") + 
           " --gridSize " + str(cell_size) + " -o " + out_path + " -r 50")
# print((stitch_cells_cli + " -p data.json " +
#            (("--surface " + input_surface) if not only_cube_cells else "") + 
#            " --gridSize " + str(cell_size) + " -o " + out_path + " -r 50"))

In [ ]:
import subprocess

print('stitch_cells_cli:', stitch_cells_cli)
print('exists:', os.path.exists(stitch_cells_cli))

cmd = [stitch_cells_cli, '-p', 'data.json']
if not only_cube_cells:
    cmd += ['--surface', input_surface]
cmd += ['--gridSize', str(cell_size), '-o', out_path, '-r', '50']

print('running:', ' '.join(cmd))
res = subprocess.run(cmd, capture_output=True, text=True)
print('returncode:', res.returncode)
print('stdout:\n', res.stdout)
print('stderr:\n', res.stderr)


In [ ]:
for c in [1, 2, 3, 4, 5, 6, 7, 8]:
    for E in [1e-3, 2e-3, 5e-3, 1e-2]:
        print(f"Processing for c={c} and E={E}")
        print((np.array(mat2geo.evaluate(nu, E)[5:-4]) * c).tolist())

In [ ]:
# To check printability of the geometry, we can create a DataFrame to summarize the minimum dimensions for different cell sizes and Young's moduli.Based on your printer nozzle size, we can determine if the geometry is printable or not.

import pandas as pd
import numpy as np

c_values = [1, 2, 3, 4, 5, 6, 7, 8, 16]
E_values = [1e-3, 2e-3, 5e-3, 1e-2, 2e-2]

df = pd.DataFrame(index=c_values, columns=E_values)

df.index.name = "Cell Size (c)"
df.columns.name = "Young's Modulus (E)"

for c in c_values:
    for E in E_values:
        arr = np.array(mat2geo.evaluate(nu, E)[5:-4]) * c
        min_dim = np.min(arr)        
        if min_dim < 0.4:
            df.loc[c, E] = "Unprintable"
        else:
            df.loc[c, E] = f"{min_dim:.4f}"

df

In [ ]:
import os

input_surface = r"C:\Users\ASUS\Desktop\input.STL"
E = 0.01
cell_size = 8
only_cube_cells = False

out_path = f"{input_surface[:-4]}_{E}_{cell_size}.obj_{only_cube_cells}.obj"
print("Expected output file:", out_path)
print("Exists:", os.path.exists(out_path))